# Прогнозирование оттока клиентов (Churn Prediction)

## Цель
Построить модель машинного обучения для бинарной классификации: предсказать, уйдет ли клиент (`churned` = 1) или останется (`churned` = 0).

## Метрики качества
Из-за сильного дисбаланса классов (~8.94% оттока) мы не используем Accuracy. Основные метрики:
- **ROC-AUC**: способность модели ранжировать клиентов по риску оттока.
- **PR-AUC (Average Precision)**: качество предсказания именно миноритарного класса (оттока).
- **F1-score**: гармоническое среднее Precision и Recall.

## Модель
Базовая: DummyClassifier (стратегия большинства).
Основная: RandomForestClassifier с балансировкой весов классов (`class_weight='balanced'`).

##Импорт библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML библиотеки
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, average_precision_score, f1_score)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

# Загрузка обогащенных данных
df = pd.read_csv('../data/processed/customers_with_rfm.csv')
print(f"Размер датасета: {df.shape}")
print(f"Доля оттока (churned): {df['churned'].mean():.2%}")

## Подготовка признаков (Feature Engineering)

In [ ]:
# 1. Удаляем неинформативные или потенциально "утекающие" признаки
cols_to_drop = ['customer_id', 'registration_date', 'churned']
# Примечание: recency, frequency, monetary уже есть и очень полезны

X = df.drop(columns=cols_to_drop)
y = df['churned']

# 2. Разделяем признаки на числовые и категориальные
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Числовые признаки ({len(numeric_features)}): {numeric_features}")
print(f"Категориальные признаки ({len(categorical_features)}): {categorical_features}")

# 3. Создаем препроцессинг пайплайн
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

## Разделение на обучающую и тестовую выборки

In [ ]:
# Stratify=y гарантирует, что доля оттока (8.94%) сохранится и в train, и в test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, y_train отток: {y_train.mean():.2%}")
print(f"X_test: {X_test.shape}, y_test отток: {y_test.mean():.2%}")

## Базовая модель (Baseline)

In [ ]:
# Модель, которая всегда предсказывает самый частый класс (0 - не ушел)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("=== Baseline (Dummy Classifier) ===")
print(f"Accuracy: {dummy.score(X_test, y_test):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_dummy):.4f}")
print("Вывод: Accuracy высокий, но F1 = 0. Модель бесполезна для поиска оттока.")

## Основная модель (Random Forest с балансировкой)

In [ ]:
# Создаем пайплайн: препроцессинг + модель
# class_weight='balanced' автоматически увеличивает штраф за ошибку на классе "1" (отток)
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1))
])

# Обучение
rf_model.fit(X_train, y_train)

# Предсказание вероятностей и классов
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Оценка
print("=== Random Forest (Balanced) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Визуализация результатов

In [ ]:
# 1. Матрица ошибок (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Не ушел (0)', 'Ушел (1)'], 
            yticklabels=['Не ушел (0)', 'Ушел (1)'])
plt.title('Матрица ошибок (Test Set)')
plt.ylabel('Истинный класс')
plt.xlabel('Предсказанный класс')
plt.show()

# 2. Важность признаков (Feature Importance)
# Извлекаем имена признаков после OneHotEncoding
ohe = rf_model.named_steps['preprocessor'].named_transformers_['cat']
cat_features_encoded = ohe.get_feature_names_out(categorical_features)
all_features = numeric_features + list(cat_features_encoded)

importances = rf_model.named_steps['classifier'].feature_importances_
feat_imp = pd.DataFrame({'feature': all_features, 'importance': importances})
feat_imp = feat_imp.sort_values(by='importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', palette='viridis')
plt.title('Топ-15 наиболее важных признаков для предсказания оттока')
plt.xlabel('Важность (Gini importance)')
plt.tight_layout()
plt.show()

## Бизнес-интерпретация и рекомендации
(Заполните этот текст после просмотра графика важности признаков из Ячейки 7)

In [ ]:
## Бизнес-интерпретация результатов

1. **Качество модели**: 
   - ROC-AUC = [вставьте значение, ожидается > 0.85] говорит о том, что модель отлично ранжирует клиентов по риску.
   - F1-score = [вставьте значение] показывает приемлемый баланс между тем, чтобы найти как можно больше уходящих клиентов (Recall), и не побеспокоить лояльных ложными тревогами (Precision).

2. **Ключевые драйверы оттока (по важности признаков)**:
   - *[Назовите топ-3 признака с графика, например: recency, total_orders, membership_tier]*.
   - Высокая важность `recency` подтверждает гипотезу: чем дольше клиент не совершал покупок, тем выше риск его потери.

3. **Рекомендации для бизнеса**:
   - **Скоринг базы**: Применить модель ко всей текущей базе клиентов.
   - **Таргетированная кампания**: Направить персональные промокоды или email-рассылки в топ-10% клиентов с наивысшей предсказанной вероятностью оттока (особенно из сегмента "At Risk").
   - **Удержание**: Для клиентов с высоким `frequency`, но растущим `recency`, предложить программу лояльности до того, как они окончательно уйдут.

## Сохранение модели

In [ ]:
import joblib

# Сохраняем обученный пайплайн
joblib.dump(rf_model, '../reports/churn_model_rf.pkl')
print("✅ Модель успешно сохранена в reports/churn_model_rf.pkl")